<font size=10>**DATA EXPLORATION & PREPROCESSING**</font> <a class="anchor" id='title'></a> 

**Bachelor's in Data Science - NOVA IMS (25/26)**

<font color='#BFD72' size=5>**Research Question: "Which companies have a dominant position within specific municipalities?"**</font> 

**Data**: 
- [*Portal BASE*](https://www.base.gov.pt/Base4/pt/pesquisa/?type=contratos&texto=&adjudicante=&adjudicataria=&tipo=2&tipocontrato=0&cpv=&aqinfo=&desdeprazoexecucao=&ateprazoexecucao=&sel_price=price_11&desdeprecocontrato=&ateprecocontrato=&desdeprecoefectivo=&ateprecoefectivo=&sel_date=date_11&desdedatacontrato=2023-01-01&atedatacontrato=2026-03-31&desdedatapublicacao=&atedatapublicacao=&desdedatafecho=&atedatafecho=&pais=0&distrito=0&concelho=0)

- [*Treated Datasets*](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/#/resources)

**Group B**
- Beatriz Marques 20231605
- Maria Inês Santos 20231630
- Luís Soeiro 20211536
- Rodrigo Silva 20231602

<font color='#BFD72' size=6>**TABLE OF CONTENTS**</font> <a class="anchor" id='toc'></a>  
- [1. Imports](#1)  
- [2. Data Integration](#2)  
- [3. Data Preprocessing](#3) 
    - [3.1 Filtering](#31-filtering)
    - [3.2 Drop Data](#32-drop-data)
    - [3.3 Data Types](#33-data-types)
    - [3.4 Text Preprocessing](#34-text-preprocessing)
    - [3.5 Outiers](#35-outiers)
    - [3.6 Missing Values](#36-missing-values)
- [4. Export Preprocessed Data](#4-export-preprocessed-data)

# <font color='#BFD72F' size=6>**1. Imports**</font> <a class="anchor" id="1"></a>

[Back to TOC](#toc)

In [1]:
import warnings
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')

In [2]:
import sys
import os

# Get the absolute path of the source_code folder
source_code_path = os.path.abspath('../source')

# Add the source_code folder to sys.path
if source_code_path not in sys.path:
    sys.path.append(source_code_path)

In [3]:
import subprocess, sys, importlib
import pandas as pd
import plotly.express as px
import re
import plotly.graph_objects as go

try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
    importlib.invalidate_caches()
    import openpyxl
import os
import shutil

# <font color='#BFD72F' size=6>**2. Data Integration**</font> <a class="anchor" id="2"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **Contract Timeframe**: From January 1, 2023 to April 25, 2026

In [4]:
# MERGE DATASETS
paths = {
#    "2023_part01": "../data/contratos2023_part01.csv",
#    "2023_part02": "../data/contratos2023_part02.csv",
#    "2023_part03": "../data/contratos2023_part03.csv",
#    "2024_part01": "../data/contratos2024_part01.csv",
#    "2024_part02": "../data/contratos2024_part02.csv",
#    "2024_part03": "../data/contratos2024_part03.csv",
    "2025_part01": "../data/contratos2025_part01.csv",
    "2025_part02": "../data/contratos2025_part02.csv",
    "2025_part03": "../data/contratos2025_part03.csv",
    "2026": "../data/contratos2026.csv",
}

datasets = {}

merged_dataset = pd.DataFrame()

for year, path in paths.items():
    print(f"Loading dataset for {year} from {path}...")
    datasets[year] = pd.read_csv(path)
    print(f"Dataset for {year} loaded successfully with shape {datasets[year].shape}.")
    merged_dataset = pd.concat([merged_dataset, datasets[year]], ignore_index=True)

print(f"Merged dataset created with shape {merged_dataset.shape}.")

Loading dataset for 2025_part01 from ../data/contratos2025_part01.csv...
Dataset for 2025_part01 loaded successfully with shape (81131, 35).
Loading dataset for 2025_part02 from ../data/contratos2025_part02.csv...
Dataset for 2025_part02 loaded successfully with shape (81132, 35).
Loading dataset for 2025_part03 from ../data/contratos2025_part03.csv...
Dataset for 2025_part03 loaded successfully with shape (81132, 35).
Loading dataset for 2026 from ../data/contratos2026.csv...
Dataset for 2026 loaded successfully with shape (69171, 35).
Merged dataset created with shape (312566, 35).


In [5]:
merged_dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 312566 entries, 0 to 312565
Data columns (total 35 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   idcontrato                312566 non-null  int64  
 1   nAnuncio                  48447 non-null   str    
 2   TipoAnuncio               48447 non-null   str    
 3   idINCM                    48447 non-null   float64
 4   tipoContrato              312566 non-null  str    
 5   idprocedimento            312566 non-null  int64  
 6   tipoprocedimento          312566 non-null  str    
 7   objectoContrato           312565 non-null  str    
 8   descContrato              312566 non-null  str    
 9   adjudicante               312566 non-null  str    
 10  adjudicatarios            312487 non-null  str    
 11  dataPublicacao            312566 non-null  str    
 12  dataCelebracaoContrato    311273 non-null  str    
 13  precoContratual           312566 non-null  float64
 14 

$\rightarrow$**Columns To Keep**:

**Identifiers & Contract Info**

| column name | |
|--- | --- |
| idcontrato | |
| tipoContrato | |
| tipoFimContrato | |
| CPV | |
| tipoprocedimento | |

**Entities**

| column name | |
|--- | --- |
| adjudicante | |
| adjudicatarios | |
| concorrentes | |

**Financial Variables**

| column name | |
|--- | --- |
| precoBaseProcedimento | |
| precoContratual | |
| PrecoTotalEfetivo | |

**Location** 

| column name | |
|--- | --- |
| LocalExecucao | |

**Dates** 

| column name | |
|--- | --- |
| dataDecisaoAdjudicacao | |
| dataCelebracaoContrato | |
| dataPublicacao | |
| dataFechoContrato | |

In [6]:
cols_to_keep = [
    'idcontrato', 'tipoContrato', 'tipoFimContrato', 'CPV', 'tipoprocedimento',
    'adjudicante', 'adjudicatarios', 'concorrentes', 
    'precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo', 
    'LocalExecucao', 
    "dataDecisaoAdjudicacao", "dataCelebracaoContrato", "dataPublicacao", "dataFechoContrato"
]

subset = merged_dataset[cols_to_keep]

In [7]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 7482 Public Entities.
There are 80949 Companies.
So, in total our analysis contains 88431 Nodes.


# <font color='#BFD72F' size=6>**3. Data Preprocessing**</font> <a class="anchor" id="3"></a>
  
[Back to TOC](#toc)

## <font size=6>**3.1 Duplicates**</font> <a class="anchor" id="3.1"></a>
  
[Back to TOC](#toc)

In [8]:
# Check duplicated rows
duplicated_rows = subset.duplicated()
print(f"Number of duplicated rows: {duplicated_rows.sum()}")

Number of duplicated rows: 1


In [9]:
# dropping duplicated rows
subset = subset.drop_duplicates()

## <font size=6>**3.2 Missing Values**</font> <a class="anchor" id="3.2"></a>
  
[Back to TOC](#toc)

In [10]:
total = len(subset)
missing_counts = subset.isnull().sum()
missing_pct = (missing_counts / total * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_pct': missing_pct
}).sort_values('missing_pct', ascending=False)

# show only columns with any missing values
missing_df

,missing_count,missing_pct
tipoFimContrato,273253,87.42
dataFechoContrato,271951,87.01
concorrentes,181045,57.92
dataDecisaoAdjudicacao,1293,0.41
dataCelebracaoContrato,1293,0.41
LocalExecucao,944,0.30
adjudicatarios,79,0.03
idcontrato,0,0.00
tipoprocedimento,0,0.00
adjudicante,0,0.00


In [11]:
# rows with missing values except for 'concorrentes', 'dataFechoContrato' and 'tipoFimContrato'
subset = subset.dropna(subset=[col for col in subset.columns if col not in ['concorrentes', 'dataFechoContrato', 'tipoFimContrato']])

In [12]:
print("Initial Number of Contracts: {}".format(merged_dataset.shape[0]))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/merged_dataset.shape[0]))*100, 2)))

Initial Number of Contracts: 312566
Number of Contracts Now: 311192
Percentage of Deleted Contracts: 0.44%


## <font size=6>**3.3 Preprocessing Per Column**</font> <a class="anchor" id="3.3"></a>
  
[Back to TOC](#toc)

### <font size=6>3.3.1 Date Columns</font> <a class="anchor" id="3.3.1"></a>
  
[Back to TOC](#toc)

In [13]:
# date type conversion    
subset["dataPublicacao"] = pd.to_datetime(subset["dataPublicacao"], errors='coerce')
subset["dataCelebracaoContrato"] = pd.to_datetime(subset["dataCelebracaoContrato"], errors='coerce')
subset["dataDecisaoAdjudicacao"] = pd.to_datetime(subset["dataDecisaoAdjudicacao"], errors='coerce')
subset["dataFechoContrato"] = pd.to_datetime(subset["dataFechoContrato"], errors='coerce')

In [14]:
dfs = []

date_cols = {
    'dataDecisaoAdjudicacao': 'Decision',
    'dataCelebracaoContrato': 'Celebration',
    'dataFechoContrato': 'Closure'
}

for col, label in date_cols.items():
    df = (
        subset
        .dropna(subset=[col])
        .assign(month=subset[col].dt.to_period('M').dt.to_timestamp())
        .groupby('month')
        .size()
        .reset_index(name='number_of_contracts')
    )
    
    df['type'] = label
    dfs.append(df)

final_df = pd.concat(dfs)

fig = px.line(
    final_df,
    x='month',
    y='number_of_contracts',
    color='type',
    title='Number of Contracts by Month (Different Dates)',
    labels={
        'month': 'Month',
        'number_of_contracts': 'Number of Contracts',
        'type': 'Date Type'
    }
)

fig.show()

In [15]:
# compute difference in months between celebration and closure, then plot histogram
months_df = subset.dropna(subset=['dataCelebracaoContrato','dataFechoContrato']).copy()
s = months_df['dataCelebracaoContrato']
e = months_df['dataFechoContrato']
months_df['months_diff'] = (e.dt.year - s.dt.year) * 12 + (e.dt.month - s.dt.month) + (e.dt.day - s.dt.day) / 30.0

fig_months = px.histogram(
    months_df,
    x='months_diff',
    nbins=100,
    title='Months between Celebration and Closure',
    labels={'months_diff': 'Months difference', 'count': 'Number of Contracts'}
)
fig_months.update_xaxes(range=[float(months_df['months_diff'].min()), float(months_df['months_diff'].max())])
fig_months.show()


In [16]:
# TODO: cut the errors

### <font size=6>3.3.2 Tipo de Procedimento</font> <a class="anchor" id="3.3.2"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **tipoprocedimento**: Concurso público

In [17]:
subset = subset[subset['tipoprocedimento'] == 'Concurso público']
subset.drop(columns=['tipoprocedimento'], inplace=True)

In [18]:
subset.shape

(47887, 15)

### <font size=6>3.3.3 Tipo de Contrato</font> <a class="anchor" id="3.3.3"></a>
  
[Back to TOC](#toc)

In [19]:
subset['tipoContrato'] = subset['tipoContrato'].astype(str).str.replace(r'[\r\n]+', ' | ', regex=True).str.strip()

In [20]:
subset['tipoContrato'].value_counts()

tipoContrato
Aquisição de bens móveis                                                            26603
Aquisição de serviços                                                               13224
Empreitadas de obras públicas                                                        6505
Locação de bens móveis                                                                890
Aquisição de bens móveis | Aquisição de serviços                                      390
Concessão de serviços públicos                                                        110
Aquisição de serviços | Locação de bens móveis                                         57
Aquisição de bens móveis | Locação de bens móveis                                      40
Aquisição de serviços | Empreitadas de obras públicas                                  16
Outros                                                                                 15
Aquisição de bens móveis | Empreitadas de obras públicas                               

In [21]:
subset_counts = subset['tipoContrato'].value_counts().reset_index()
subset_counts.columns = ['tipoContrato', 'count']

# Get top 10
top_10 = subset_counts.head(10)

fig = px.bar(
    top_10,
    x='tipoContrato',
    y='count',
    title='Number of Contracts by Contract Type',
    labels={'tipoContrato': 'Contract Type', 'count': 'Number of Contracts'}
)

fig.show()

### <font size=6>3.3.5 Concorrentes</font> <a class="anchor" id="3.3.5"></a>
  
[Back to TOC](#toc)

In [22]:
subset['concorrentes'] = (
    subset['concorrentes']
    .astype(str)
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # remove excessive spaces
    .str.replace(r'\s+', ' ', regex=True)
    # remove trailing numbers (like " 102", " 89", etc.)
    .str.replace(r'\s+\d+\s*$', '', regex=True)
    # final trim
    .str.strip()
)

In [23]:
# number of competitors per contract
subset['nr_concorrentes'] = subset['concorrentes'].apply(
    lambda x: len([i for i in re.split(r'\s*\|\s*', x) if i]) 
    if isinstance(x, str) else 0
)

fig = px.histogram(
    subset,
    x='nr_concorrentes',
    nbins=30,
    title='Histogram of Number of Competitors'
)

fig.show()

### <font size=6>3.3.6 Adjudicante & Adjudicatário</font> <a class="anchor" id="3.3.6"></a>
  
[Back to TOC](#toc)

In [24]:
# extract contribuinte numbers from adjudicante and adjudicatarios
subset['contribuinte_adjudicante'] = subset['adjudicante'].str.extract(r'(\d{9})')
subset['adjudicante'] = subset['adjudicante'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

subset['contribuinte_adjudicatarios'] = subset['adjudicatarios'].str.extract(r'(\d{9})')
subset['adjudicatarios'] = subset['adjudicatarios'].str.replace(r'\s*\d{9}\s* - ', '', regex=True).str.strip()

### <font size=6>3.3.7 Local de Execução</font> <a class="anchor" id="3.3.7"></a>
  
[Back to TOC](#toc)

$\rightarrow$ **district**: Lisboa

In [25]:
# LocalExecucao
subset['LocalExecucao'] = subset['LocalExecucao'].fillna('')

subset['LocalExecucao'] = (
    subset['LocalExecucao']
    # replace line breaks with separator
    .str.replace(r'[\r\n]+', ' | ', regex=True)
    # normalize spaces
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    # remove duplicates inside each cell
    .apply(lambda x: ' | '.join(dict.fromkeys(x.split(' | '))) if x else x)
)

first_location = subset['LocalExecucao'].str.split(' \| ', expand=False).str[0]

split_cols = first_location.str.split(', ', expand=True)

split_cols = split_cols.reindex(columns=[0, 1, 2])
split_cols.columns = ['country', 'district', 'city']

subset[['country', 'district', 'city']] = split_cols

for col in ['country', 'district', 'city']:
    subset[col] = subset[col].replace(r'^\s*$', pd.NA, regex=True)

# handle inconsistent structures
n_parts = first_location.str.split(', ').str.len()

# if only 1 part - it's country
subset.loc[n_parts == 1, ['district', 'city']] = pd.NA

# if 2 parts - assume country + district
subset.loc[n_parts == 2, 'city'] = pd.NA

In [26]:
subset[['country', 'district', 'city']].value_counts(dropna=False)

country   district    city    
Portugal  NaN         NaN         8947
          Lisboa      Lisboa      8002
          Porto       Porto       1991
          Faro        Faro         946
          Coimbra     Coimbra      935
                                  ... 
          Portalegre  Gavião         1
Ucrânia   NaN         NaN            1
Portugal  Beja        Alvito         1
          Portalegre  Monforte       1
Alemanha  NaN         NaN            1
Name: count, Length: 338, dtype: int64

In [27]:
subset_plot = (
    subset.dropna(subset=["district"])  # remove missing districts
      .groupby("district")
      .agg(
          n_contracts=("idcontrato", "count"),
          n_adjudicantes=("contribuinte_adjudicante", "nunique"),
          n_adjudicatarios=("contribuinte_adjudicatarios", "nunique")
      )
      .reset_index()
)

In [28]:
fig = go.Figure()

# --- traces ---
part01 = subset_plot.sort_values("n_contracts", ascending=False)
fig.add_trace(go.Bar(
    x=part01["district"],
    y=part01["n_contracts"],
    name="Contracts",
    visible=True  # default visible
))

part02 = subset_plot.sort_values("n_adjudicantes", ascending=False)
fig.add_trace(go.Bar(
    x=part02["district"],
    y=part02["n_adjudicantes"],
    name="Adjudicantes",
    visible=False
))

part03 = subset_plot.sort_values("n_adjudicatarios", ascending=False)
fig.add_trace(go.Bar(
    x=part03["district"],
    y=part03["n_adjudicatarios"],
    name="Adjudicatarios",
    visible=False
))

# --- dropdown ---
fig.update_layout(
    updatemenus=[
        dict(
            buttons=[
                dict(
                    label="Contracts",
                    method="update",
                    args=[{"visible": [True, False, False]},
                          {"title": "Number of Contracts"}]
                ),
                dict(
                    label="Adjudicantes",
                    method="update",
                    args=[{"visible": [False, True, False]},
                          {"title": "Number of Adjudicantes"}]
                ),
                dict(
                    label="Adjudicatarios",
                    method="update",
                    args=[{"visible": [False, False, True]},
                          {"title": "Number of Adjudicatarios"}]
                ),
            ],
            direction="down",
            showactive=True
        )
    ]
)

fig.update_layout(
    title="Contracts per District",
    xaxis_title="District",
    yaxis_title="Count",
    xaxis_tickangle=-45
)

fig.show()

In [29]:
subset = subset[subset['district'] == 'Lisboa']
subset.drop(columns=['LocalExecucao', 'country', 'district'], inplace=True)

In [30]:
subset['city'].value_counts()

city
Lisboa                    8002
Loures                     872
Sintra                     811
Cascais                    683
Oeiras                     637
Amadora                    296
Vila Franca de Xira        233
Mafra                      167
Odivelas                   103
Torres Vedras               74
Alenquer                    52
Arruda dos Vinhos           31
Lourinhã                    23
Cadaval                     18
Azambuja                    11
Sobral de Monte Agraço       9
Name: count, dtype: int64

### <font size=6>3.3.8 Price Columns</font> <a class="anchor" id="3.3.8"></a>
  
[Back to TOC](#toc)

In [31]:
# keep idcontrato
price_cols = ['precoBaseProcedimento', 'precoContratual', 'PrecoTotalEfetivo']
box_df = subset[['idcontrato'] + price_cols].copy()

# convert only price columns
box_df[price_cols] = box_df[price_cols].apply(pd.to_numeric, errors='coerce')

# melt while keeping idcontrato
long_df = box_df.melt(
    id_vars='idcontrato',
    var_name='Column',
    value_name='Value'
).dropna()

fig = px.box(
    long_df,
    x='Column',
    y='Value',
    color='Column',
    points='outliers',
    title='Distribution of Price Columns',
    hover_data=['idcontrato'] 
)

fig.update_layout(showlegend=False)
fig.show()

In [32]:
# TODO: delete rows where precoContratual is zero or negative, as they are likely errors

In [33]:
subset.info()

<class 'pandas.DataFrame'>
Index: 12393 entries, 58 to 312485
Data columns (total 18 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   idcontrato                   12393 non-null  int64         
 1   tipoContrato                 12393 non-null  str           
 2   tipoFimContrato              691 non-null    str           
 3   CPV                          12393 non-null  str           
 4   adjudicante                  12393 non-null  str           
 5   adjudicatarios               12393 non-null  str           
 6   concorrentes                 6493 non-null   str           
 7   precoBaseProcedimento        12393 non-null  float64       
 8   precoContratual              12393 non-null  float64       
 9   PrecoTotalEfetivo            12393 non-null  float64       
 10  dataDecisaoAdjudicacao       12393 non-null  datetime64[us]
 11  dataCelebracaoContrato       12393 non-null  datetime64

In [34]:
# Ensure numeric conversion
subset['precoContratual'] = pd.to_numeric(
    subset['precoContratual'],
    errors='coerce'
)

# Filter rows under 50k
under_50k = subset[subset['precoContratual'] < 20000]

# Show result
print(f"Rows with precoContratual < 50,000: {len(under_50k)}")

under_50k[
    [
        'idcontrato',
        'precoContratual',
        'precoBaseProcedimento',
        'PrecoTotalEfetivo',
        'adjudicante',
        'adjudicatarios'
    ]
].head(50)

Rows with precoContratual < 50,000: 4951


,idcontrato,precoContratual,precoBaseProcedimento,PrecoTotalEfetivo,adjudicante,adjudicatarios
452,11132636,15742.65,58000.00,0.00,Centro de Formação Profissional da Indústria d...,"Ageas Portugal, Companhia de Seguros, SA"
735,11141114,2088.00,105914.40,0.00,Município de Sintra,"UpElev-Elevadores e Engenharia, Lda."
817,11139751,10650.00,125320.00,0.00,"Parques de Sintra - Monte da Lua, S. A.",- - PEDRO RODRIGUES DA COSTA
860,11144492,6338.80,134973.00,7142.35,Serviços Sociais da Guarda Nacional Republicana,dragondisplay unipessoal lda
955,11146952,80.00,8373.98,0.00,"Unidade Local de Saúde de Santa Maria, E. P. E.","CSPSaúde,"
992,11143130,16805.76,64999.99,0.00,Município de Sintra,"Margem Mítica-Manutenção e Reabilitação, Unipe..."
1138,11148135,6329.88,125320.00,0.00,"Parques de Sintra - Monte da Lua, S. A.",- - Pedro Miguel Gonçalves Dias
1174,11148587,4433.40,8373.98,0.00,"Unidade Local de Saúde de Santa Maria, E. P. E.","Bio-Rad Laboratories, Lda."
1209,11148860,697.50,27825.00,0.00,Santa Casa da Misericórdia de Lisboa,"Fresenius Kabi Pharma Portugal, Lda."
1294,11154716,7700.00,45300.00,0.00,Secretaria-Geral do Ministério da Defesa Nacional,"CRC - Car Rental Company, Lda."


In [35]:

# Ensure numeric
subset['precoContratual'] = pd.to_numeric(
    subset['precoContratual'],
    errors='coerce'
)

# --- IQR method ---
Q1 = subset['precoContratual'].quantile(0.25)
Q3 = subset['precoContratual'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Find outliers
outliers_df = subset[
    (subset['precoContratual'] < lower_bound) |
    (subset['precoContratual'] > upper_bound)
]

print(f"Number of outliers: {len(outliers_df)}")
print(f"Lower bound: {lower_bound:,.2f}")
print(f"Upper bound: {upper_bound:,.2f}")

# Show relevant columns
outliers_df[
    [
        'idcontrato',
        'precoContratual',
        'precoBaseProcedimento',
        'PrecoTotalEfetivo',
        'adjudicante',
        'adjudicatarios'
    ]
].sort_values('precoContratual', ascending=False)

Number of outliers: 1445
Lower bound: -195,374.95
Upper bound: 341,224.97


,idcontrato,precoContratual,precoBaseProcedimento,PrecoTotalEfetivo,adjudicante,adjudicatarios
100131,11577383,40000000.00,0.00,0.0,"Instituto do Turismo de Portugal, I. P.","DENTSU CREATIVE - Agência de Publicidade, S,A."
59237,11393908,31147209.54,33385135.05,0.0,Município de Lisboa,Gertal - Companhia Geral de Restaurantes e Ali...
3085,11176543,29947140.00,29947140.00,0.0,Direção-Geral de Alimentação e Veterinária,ITSLLF
221516,12198094,29462012.00,35152375.10,0.0,"Companhia Carris de Ferro de Lisboa, EM, SA",BP
280564,14248029,28696879.04,24000000.00,0.0,"Águas do Tejo Atlântico, SA","Luságua - Serviços Ambientais, SA"
...,...,...,...,...,...,...
144008,11639432,342800.00,745211.75,0.0,"Unidade Local de Saúde de Loures-Odivelas, EPE",Medarthrex Unipessoal Lda
16475,11254295,342601.20,347367.65,0.0,SPMS - Serviços Partilhados do Ministério da S...,"Fine Facility Services, Lda"
295791,14504736,342329.88,342329.88,0.0,Serviço de Utilização Comum dos Hospitais,- - INFORMATICA EL CORTE INGLES
26642,11196773,342145.84,344656.00,0.0,Polícia de Segurança Pública,"MultiTrab – Serviços, Lda."


In [36]:
# TODO: histogram of price distribution with log scale, showing outliers in different color

### <font size=6>3.3.4 CPV</font> <a class="anchor" id="3.3.4"></a>
  
[Back to TOC](#toc)

In [37]:
#subset['CPV'] = subset['CPV'].astype(str).str.replace('\n', ' | ', regex=True).str.strip()
#subset['CPV'] = subset['CPV'].astype(str).str.replace(r'\s*\d{8}-\d\s*-\s*', ' ', regex=True).str.strip()

In [38]:
subset['CPV'].value_counts() 

CPV
33140000-3 - Material médico de consumo                           2613
33600000-6 - Produtos farmacêuticos                                540
33100000-1 - Equipamento médico                                    482
33696500-0 - Reagentes de laboratório                              255
33690000-3 - Medicamentos vários                                   182
                                                                  ... 
31681400-7 - Componentes eléctricos                                  1
30197643-5 - Papel para fotocópia                                    1
37800000-6 - Material para actividades artísticas e artesanato       1
43310000-9 - Máquinas para engenharia civil                          1
33181520-3 - Produtos de consumo para hemodiálise renal              1
Name: count, Length: 1507, dtype: int64

In [39]:
subset_counts = subset['CPV'].value_counts().reset_index()
subset_counts.columns = ['CPV', 'count']

# Get top 20
top_20 = subset_counts.head(20)

fig = px.bar(
    top_20,
    x='count',
    y='CPV',
    orientation='h',
    title='Top 20 CPVs by Number of Contracts',
    labels={'CPV': 'CPV', 'count': 'Number of Contracts'},
    width=1700,   
    height=800  
)

# Largest bar on top
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

In [40]:
# Ensure CPV is string (important)
subset['CPV'] = subset['CPV'].astype(str)

# Extract first 2 digits
subset['cpv_prefix'] = subset['CPV'].str[:2]

# Mapping dictionary (based on your table)
cpv_map = {
    "03": "Agricultura, pesca e silvicultura",
    "09": "Energia e combustíveis",
    "14": "Mineração e metais",
    "15": "Alimentação, bebidas e tabaco",
    "16": "Maquinaria agrícola",
    "18": "Vestuário e acessórios",
    "19": "Têxteis, couro, plástico e borracha",
    "22": "Material impresso",
    "24": "Produtos químicos",
    "30": "Equipamento de escritório e informática",
    "31": "Equipamento elétrico e iluminação",
    "32": "Telecomunicações",
    "33": "Equipamento médico e farmacêutico",
    "34": "Equipamento de transporte",
    "35": "Segurança e defesa",
    "37": "Desporto, jogos e artesanato",
    "38": "Equipamento laboratorial e óptico",
    "39": "Mobiliário e limpeza",
    "41": "Água tratada",
    "42": "Máquinas industriais",
    "43": "Construção e extração",
    "44": "Materiais de construção",
    "45": "Construção",
    "48": "Software e sistemas de informação",
    "50": "Reparação e manutenção",
    "51": "Instalação",
    "55": "Hotelaria e restauração",
    "60": "Transporte",
    "63": "Serviços auxiliares de transporte",
    "64": "Telecomunicações postais",
    "65": "Serviços públicos",
    "66": "Finanças e seguros",
    "70": "Imobiliário",
    "71": "Arquitetura e engenharia",
    "72": "TI e consultoria",
    "73": "Investigação e desenvolvimento",
    "75": "Administração pública e segurança social",
    "76": "Indústria de petróleo e gás",
    "77": "Agricultura e serviços relacionados",
    "79": "Serviços empresariais",
    "80": "Educação",
    "85": "Saúde e ação social",
    "90": "Ambiente e resíduos",
    "92": "Cultura e desporto",
    "98": "Outros serviços"
}

# Create aggregated CPV category
subset['agg_cpv'] = subset['cpv_prefix'].map(cpv_map).fillna("Outros / Não classificado")

In [41]:
subset_counts = subset['agg_cpv'].value_counts().reset_index()
subset_counts.columns = ['agg_cpv', 'count']

# Get top 20
top_20 = subset_counts.head(20)

fig = px.bar(
    top_20,
    x='count',
    y='agg_cpv',
    orientation='h',
    title='Top 20 CPVs by Number of Contracts',
    labels={'agg_cpv': 'agg_cpv', 'count': 'Number of Contracts'},
    width=1700,   
    height=800  
)

# Largest bar on top
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'}
)

fig.show()

## <font size=6>**3.4 Final Data**</font> <a class="anchor" id="3.4"></a>
  
[Back to TOC](#toc)

In [42]:
subset.head()

,idcontrato,tipoContrato,tipoFimContrato,CPV,adjudicante,adjudicatarios,concorrentes,precoBaseProcedimento,precoContratual,PrecoTotalEfetivo,dataDecisaoAdjudicacao,dataCelebracaoContrato,dataPublicacao,dataFechoContrato,nr_concorrentes,contribuinte_adjudicante,contribuinte_adjudicatarios,city,cpv_prefix,agg_cpv
58,11124346,Aquisição de serviços,NaN,90420000-7 - Serviços de tratamento de águas r...,"Tratolixo - Tratamento de Residuos Sólidos, E....",Carmona,"516510657-Lucena &amp; Lucena, Lda | 501741380...",549920.0,549920.00,0.0,2024-11-15,2025-01-02,2025-01-02,NaT,2,502444010,501741380,Mafra,90,Ambiente e resíduos
66,11123851,Aquisição de serviços,NaN,85111900-9 - Serviços de diálise hospitalar,"Unidade Local de Saúde de Lisboa Ocidental, E....","Fresenius Medical Care Portugal, SA",NaN,310000.0,300000.00,0.0,2024-11-21,2025-01-02,2025-01-02,NaT,0,507618319,503070220,Oeiras,85,Saúde e ação social
116,11124453,Aquisição de serviços,NaN,66510000-8 - Serviços de seguros,Ciência Viva - Agência Nacional para a Cultura...,"JOÃO MATA Lda.Willis - Corretores de Seguros, ...","500188629-Willis - Corretores de Seguros, S.A....",160000.0,152405.91,0.0,2024-12-13,2025-01-02,2025-01-02,NaT,4,504300156,500150141,Lisboa,66,Finanças e seguros
163,11126382,Aquisição de serviços,NaN,72253100-4 - Serviços de help-desk,"EPAL - Empresa Portuguesa das Águas Livres, S. A.","Alpibre, Lda","507902238-Shore Spun, Lda, | 515931217-Alpibre...",535000.0,214576.56,0.0,2024-04-24,2025-01-02,2025-01-03,NaT,10,500906840,515931217,Lisboa,72,TI e consultoria
192,11127239,Aquisição de serviços,NaN,72268000-1 - Serviços de fornecimento de software,"Cascais Próxima - Gestão de Mobilidade, Espaço...","ACIN - iCloud Solutions, Lda","511135610-ACIN - iCloud Solutions, Lda",74520.0,74019.96,0.0,2024-12-05,2025-01-02,2025-01-03,NaT,1,504853635,511135610,Cascais,72,TI e consultoria


In [43]:
subset.info()

<class 'pandas.DataFrame'>
Index: 12393 entries, 58 to 312485
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   idcontrato                   12393 non-null  int64         
 1   tipoContrato                 12393 non-null  str           
 2   tipoFimContrato              691 non-null    str           
 3   CPV                          12393 non-null  str           
 4   adjudicante                  12393 non-null  str           
 5   adjudicatarios               12393 non-null  str           
 6   concorrentes                 6493 non-null   str           
 7   precoBaseProcedimento        12393 non-null  float64       
 8   precoContratual              12393 non-null  float64       
 9   PrecoTotalEfetivo            12393 non-null  float64       
 10  dataDecisaoAdjudicacao       12393 non-null  datetime64[us]
 11  dataCelebracaoContrato       12393 non-null  datetime64

In [44]:
print("There are {} Public Entities.".format(subset['adjudicante'].nunique()))
print("There are {} Companies.".format(subset['adjudicatarios'].nunique()))

print("So, in total our analysis contains {} Nodes.".format(
    subset['adjudicatarios'].nunique() + 
    subset['adjudicante'].nunique()))

There are 672 Public Entities.
There are 4763 Companies.
So, in total our analysis contains 5435 Nodes.


In [45]:
print("Initial Number of Contracts: {}".format(merged_dataset.shape[0]))
print("Number of Contracts Now: {}".format(subset.shape[0]))
print("Percentage of Deleted Contracts: {}%".format(round((1 - (subset.shape[0]/merged_dataset.shape[0]))*100, 2)))

Initial Number of Contracts: 312566
Number of Contracts Now: 12393
Percentage of Deleted Contracts: 96.04%


# <font color='#BFD72F' size=6>**4. Export Preprocessed Data**</font> <a class="anchor" id="4"></a>
  
[Back to TOC](#toc)

In [46]:
subset.to_csv("../data/preprocessed_data.csv", index=False)

In [47]:
subset.to_csv("../data/preprocessed_data2.csv", index=False)